# Lab Exercise: Rules vs. Learning
This notebook demonstrates the fundamental difference between explicit rule-based programming and a machine learning model using a decision tree on a sample cat/dog classification task.

### Objectives:
1. See how hand-picked thresholds fail on new/unseen data.
2. See how a Machine Learning algorithm discovers hidden features and optimal splits autonomously.

In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier, export_text

FEATURES = ["has_wings", "num_legs", "weight_kg", "snout_cm", "retractable_claws"]

# Training dataset (small, representative sample)
X_train = np.array([
    [0, 4,  3.5, 2.8, 1],   # cat - small
    [0, 4,  4.0, 3.0, 1],   # cat
    [0, 4,  5.2, 3.2, 1],   # cat
    [0, 4, 11.0, 3.4, 1],   # cat - maine coon, heavy
    [0, 4,  2.5, 7.5, 0],   # dog - chihuahua, light
    [0, 4, 18.0, 8.0, 0],   # dog
    [0, 4, 25.0, 9.0, 0],   # dog
    [0, 4, 32.0, 11.5, 0],  # dog
    [1, 2,  0.4, 2.0, 0],   # bird
    [1, 2,  0.9, 2.4, 0],   # bird
])
y_train = np.array(["cat", "cat", "cat", "cat",
                    "dog", "dog", "dog", "dog", "bird", "bird"])

### 1. The Rule-Based Approach
Here, a human writes the classification logic by guessing thresholds (like `weight_kg < 10` for small animals/cats).

In [ ]:
def rule_based(row):
    has_wings, legs, weight, snout, claws = row
    if has_wings:
        return "bird"
    if weight < 10:          # human guess: "small means cat"
        return "cat"
    return "dog"

def accuracy(predict, X, y):
    return float(np.mean([predict(r) == t for r, t in zip(X, y)]))

print(f"Rule-based Training Accuracy: {accuracy(rule_based, X_train, y_train):.0%}")

### 2. The Machine Learning Approach
We train a Decision Tree on the raw features, giving it no rules—only the examples.

In [ ]:
tree = DecisionTreeClassifier(random_state=0).fit(X_train, y_train)
predict_tree = lambda r: tree.predict([r])[0]

print(f"Decision Tree Training Accuracy: {accuracy(predict_tree, X_train, y_train):.0%}")
print("\n--- What the Tree Taught Itself ---")
print(export_text(tree, feature_names=FEATURES).rstrip())

### 3. Testing on Unseen Data
Now let's evaluate both approaches on 4 animals neither had ever seen before. This includes a heavy cat (13 kg) and a light dog (8 kg) to test generalizability.

In [ ]:
X_new = np.array([
    [0, 4,  6.0, 3.1, 1],   # cat
    [0, 4,  8.0, 8.5, 0],   # dog  - mid-size, under the human's 10 kg line
    [0, 4, 13.0, 3.5, 1],   # cat  - over the human's 10 kg line
    [1, 2,  1.2, 2.2, 0],   # bird
])
y_new = np.array(["cat", "dog", "cat", "bird"])
labels = ["cat   6.0 kg", "dog   8.0 kg", "cat  13.0 kg", "bird  1.2 kg"]

print(f"  {'animal':<15}{'truth':<8}{'rules':<9}{'tree':<8}")
for label, row, truth in zip(labels, X_new, y_new):
    r, t = rule_based(row), predict_tree(row)
    print(f"  {label:<15}{truth:<8}"
          f"{r + ('' if r == truth else '  X'):<9}"
          f"{t + ('' if t == truth else '  X'):<8}")

print(f"\nRule-based Test Accuracy: {accuracy(rule_based, X_new, y_new):.0%}")
print(f"Decision Tree Test Accuracy: {accuracy(predict_tree, X_new, y_new):.0%}")

### 4. Lab Experiments (Things to Try)

#### Experiment 1: Delete `retractable_claws`
1. Scroll back up and remove `retractable_claws` from `FEATURES` and index 4 from `X_train` / `X_new`.
2. Re-fit and run the notebook again.
3. Notice how the tree is forced to split on body weight and fails exactly like the human's rules!

#### Experiment 2: Unrepresentative Training Sample
1. Delete the Maine Coon (index 3) and Chihuahua (index 4) rows from training.
2. Re-fit and test. This illustrates why representative data matters more than fancy models!